In [ ]:
/**
 * DATABASE SCALING — Sharding: Splitting Data Across Multiple Databases
 * 1.  Why Sharding? .................... When read replicas are not enough
 * 2.  What is Sharding? ................ Splitting data across databases
 * 3.  Sharding vs Replication .......... Copy vs divide
 * 4.  Sharding vs Partitioning ......... Across instances vs within one
 * 5.  Practical Example ................ Sharding orders by country
 * 6.  Pros of Sharding ................. Benefits and advantages
 * 7.  Cons of Sharding ................. Complexity and tradeoffs
 * 8.  Shard Key ........................ The most important design decision
 * 9.  Shard Skew ....................... When distribution goes wrong
 * 10. Planning for Sharding ............ Do it before the ceiling hits
 * 11. Golden Rules ..................... Key principles to remember
 */

In [ ]:
/**
 * Part-1: WHY SHARDING?
 */

In [ ]:
/**
 * When Read Replicas Are Not Enough
 * ────────────────────────────────────
 *
 *   Read replicas reads distribute karte hain — lekin writes ka kya?
 *   Ek hi PRIMARY writer ki bhi limit hoti hai.
 *   Jab storage aur writes grow karte hain ek machine se zyada — SHARDING chahiye.
 *
 * WE'VE ALREADY COVERED:
 *   → Reads bottleneck → use READ REPLICAS
 *   → Single DB under pressure → first OPTIMIZE:
 *       Better schema, queries, indexes, workload distribution
 *
 * BUT WHAT HAPPENS WHEN EVEN THAT IS NOT ENOUGH?
 *   Read replicas help with read traffic, but they do NOT solve:
 *   → A single writer eventually hitting limits
 *   → Storage growing without bound on one machine
 *   → Maintenance on one HUGE database becoming riskier
 *
 * WHY READ REPLICAS CAN'T SOLVE WRITE SCALING:
 *   Reads are easy to distribute — copy the data and serve from many copies.
 *
 *   Writes are DIFFERENT. They CANNOT be "copy and distribute" because:
 *   → There must be a clear source of truth
 *   → Storage keeps growing
 *   → Write throughput on one machine has a LIMIT
 *   → Maintenance on one huge DB becomes riskier over time
 *
 * WHEN TO CONSIDER SHARDING:
 *   → Write traffic is very high
 *   → Storage is growing aggressively
 *   → One primary is becoming too large or too busy
 *   → Adding more replicas is not enough
 *
 *   At that point, you need to SPLIT THE ACTUAL DATA ITSELF.
 *   That is what sharding does.
 *
 *   THE CORE QUESTION:
 *   "Can I keep all this data and all this write load on
 *    one machine forever?"
 *
 *   For very large systems, the answer is often NO.
 */

In [ ]:
/**
 * Part-2: WHAT IS SHARDING?
 */

In [ ]:

/**
 * Data Ko Multiple Databases Mein Split Karna
 *
 * Hinglish:
 *   Replicas mein SAARA data har copy mein hota hai.
 *   Sharding mein data TUKDON mein split hota hai — har database ka apna slice.
 *   Ye ek MAJOR architectural shift hai.
 *
 * WITH REPLICAS (every replica has the same data):
 *
 *   ┌─────────────────────────────────────────────────────────────┐
 *   │                    PRIMARY                                  │
 *   │              (all rows: US, IN, EU, ...)                    │
 *   └──────────────────────┬──────────────────────────────────────┘
 *                          │
 *           ┌──────────────┼──────────────┐
 *           ▼              ▼              ▼
 *     ┌──────────┐   ┌──────────┐   ┌──────────┐
 *     │ REPLICA  │   │ REPLICA  │   │ REPLICA  │
 *     │(all rows)│   │(all rows)│   │(all rows)│
 *     └──────────┘   └──────────┘   └──────────┘
 *
 * WITH SHARDING (each shard has DIFFERENT data):
 *
 *   ┌──────────────────┐  ┌──────────────────┐  ┌──────────────────┐
 *   │     SHARD 1      │  │     SHARD 2      │  │     SHARD 3      │
 *   │   US orders      │  │   India orders   │  │   EU orders      │
 *   │   (slice)        │  │   (slice)        │  │   (slice)        │
 *   └──────────────────┘  └──────────────────┘  └──────────────────┘
 *
 * KEY DIFFERENCE:
 *   Replicas  → every database has ALL rows
 *   Sharding  → each database has ONLY A SLICE of rows
 *
 * With replicas:
 *   "One main DB has everything, and replicas COPY it."
 *
 * With sharding:
 *   "Multiple databases each OWN different parts of the dataset."
 *
 * This is a MAJOR architectural shift.
 */

In [ ]:


/**
 * Part-3: SHARDING VS REPLICATION
 */

/**
 * Copy vs Divide — Simple Way to Remember
 * ──────────────────────────────────────────
 *
 *   REPLICAS COPY karte hain (same data everywhere)
 *   SHARDS DIVIDE karte hain (different data in each)
 *   Use case bhi alag hai — reads vs writes+storage.
 *
 * ┌─────────────────┬────────────────────────┬────────────────────────────┐
 * │ ASPECT          │ REPLICATION            │ SHARDING                   │
 * ├─────────────────┼────────────────────────┼────────────────────────────┤
 * │ What happens    │ Copy same data to      │ Split data across          │
 * │                 │ multiple DBs           │ multiple DBs               │
 * ├─────────────────┼────────────────────────┼────────────────────────────┤
 * │ Data in each DB │ All rows (full dataset)│ Only a slice               │
 * ├─────────────────┼────────────────────────┼────────────────────────────┤
 * │ Primary purpose │ Scale reads            │ Scale writes and storage   │
 * ├─────────────────┼────────────────────────┼────────────────────────────┤
 * │ Write capacity  │ Still limited by       │ Distributed across         │
 * │                 │ single primary         │ multiple shards            │
 * ├─────────────────┼────────────────────────┼────────────────────────────┤
 * │ Storage         │ One primary holds all  │ Distributed across shards  │
 * ├─────────────────┼────────────────────────┼────────────────────────────┤
 * │ Complexity      │ Lower                  │ Much higher                │
 * └─────────────────┴────────────────────────┴────────────────────────────┘
 *
 * WHEN TO USE EACH:
 *   Replication  → when READS are the bottleneck
 *   Sharding     → when WRITES or STORAGE become the bottleneck
 */


/**
 * Part-4: SHARDING VS PARTITIONING
 */

/**
 * Ek DB Ke Andar vs Multiple DBs
 * ────────────────────────────────
 *
 *   Partitioning = ek hi database ke andar data ko slices mein divide karna.
 *   Sharding = multiple databases ke across data divide karna.
 *   Dono alag problems solve karte hain.
 *
 * PARTITIONING:
 *   → Happens INSIDE a single database instance
 *   → Managed by the database engine itself
 *   → Used to improve query efficiency
 *   → Still ONE database server
 *   → Easier joins and coordination
 *   → Simpler operationally
 *
 * SHARDING:
 *   → Happens ACROSS multiple database instances
 *   → Requires application/routing logic
 *   → Used to scale writes and storage
 *   → MULTIPLE database servers
 *   → Cross-shard operations become hard
 *   → Much higher operational complexity
 *
 * USE PARTITIONING WHEN:
 *   → Data still fits within one server
 *   → Queries are slow because they scan too much data
 *   → You want better efficiency without moving to multiple DBs
 *
 * USE SHARDING WHEN:
 *   → One server is no longer enough
 *   → Storage or write load has outgrown single-machine limits
 *   → True horizontal scale is needed
 *
 *   Partitioning improves organization INSIDE one DB.
 *   Sharding changes the WHOLE architecture.
 */


/**
 * Part-5: PRACTICAL EXAMPLE
 */

/**
 * Orders Table Ko Country Ke Hisaab Se Shard Karna
 * ────────────────────────────────────────────────────
 *
 *   Company USA mein start hoti hai, phir India aur Europe expand karti hai.
 *   Orders table ENORMOUS ho jaati hai — billions of rows.
 *   Even replicas ke baad bhi ek writer kaafi nahi.
 *   Solution: country ke hisaab se data shard karo.
 *
 * THE SCENARIO:
 *   → Company starts in USA
 *   → Expands to India, Europe, other regions
 *   → Orders table: billions of rows
 *   → Writes happening 24/7 from many countries
 *   → Even after optimization and replicas — ONE WRITER is not enough
 *
 * BEFORE SHARDING (one database has everything):
 *
 *   ┌──────────────────────────────────────────────────────────┐
 *   │                    ORDERS TABLE                          │
 *   │   US order 1, US order 2                                 │
 *   │   India order 1, India order 2                           │
 *   │   EU order 1, EU order 2                                 │
 *   │   ... billions more ...                                  │
 *   └──────────────────────────────────────────────────────────┘
 *
 * AFTER SHARDING (split by country):
 *
 *   ┌──────────────────┐  ┌──────────────────┐  ┌──────────────────┐
 *   │   SHARD 1        │  │   SHARD 2        │  │   SHARD 3        │
 *   │   US orders only │  │   India orders   │  │   EU orders only │
 *   │                  │  │   only           │  │                  │
 *   │   US order 1     │  │   India order 1  │  │   EU order 1     │
 *   │   US order 2     │  │   India order 2  │  │   EU order 2     │
 *   │   ...            │  │   ...            │  │   ...            │
 *   └──────────────────┘  └──────────────────┘  └──────────────────┘
 *
 * WHAT THIS ACHIEVES:
 *   → US write traffic    → hits shard 1
 *   → India write traffic → hits shard 2
 *   → EU write traffic    → hits shard 3
 *
 *   Result:
 *   → Write load no longer concentrated in one place
 *   → Storage is distributed
 *   → More storage runway
 *   → More write runway
 */


/**
 * Part-6: PROS OF SHARDING
 */

/**
 * Sharding ke Benefits
 * ──────────────────────
 *
 *   Sharding kab beneficial hota hai? Ye 5 main advantages hain.
 *
 *  1. MORE STORAGE RUNWAY
 *     Ek database ko sab kuch hold nahi karna.
 *     Data spread across shards → total storage capacity significantly increases.
 *
 *  2. MORE WRITE RUNWAY
 *     Write traffic distributed across shards.
 *     One node no longer handles every write.
 *     This is one of the BIGGEST reasons sharding exists.
 *
 *  3. ONE NODE NO LONGER CARRIES EVERYTHING
 *     Operationally ye bahut help karta hai.
 *     One overloaded node no longer responsible for full system scale.
 *     Pressure ka concentration reduce hota hai.
 *
 *  4. BETTER FAULT ISOLATION
 *     If one shard has a problem:
 *       → Only part of the data is affected
 *       → Only one segment of traffic is impacted
 *       → NOT necessarily the whole world
 *
 *     MUCH better than one giant all-powerful primary
 *     being the only place everything lives.
 *
 *  5. GEOGRAPHICAL OPTIMIZATION
 *     Shards can be placed closer to where data originates.
 *     Example: India shard in India → lower latency for Indian users.
 */


/**
 * Part-7: CONS OF SHARDING
 */

/**
 * Sharding ke Complexity aur Tradeoffs
 * ──────────────────────────────────────
 *
 *   Sharding whiteboard pe cool lagta hai — lekin real mein bahut bhaaree hai.
 *   Ye 4 major downsides hain jo production mein feel hote hain.
 *
 *  1. ROUTING LOGIC BECOMES YOUR RESPONSIBILITY
 *     Data split hone ke baad application/routing layer ko PATA HONA CHAHIYE:
 *       → Which shard to query
 *       → Which shard to write to
 *       → How to find the right shard for a given request
 *
 *     That logic must be designed, implemented, tested, and maintained.
 *
 *  2. REBALANCING IS PAINFUL
 *     Suppose one shard becomes too large or too hot.
 *     Now you may need to move data AGAIN:
 *       → Repartitioning
 *       → Migration
 *       → Downtime or careful live migration
 *       → More routing changes
 *       → Lots of operational work
 *
 *     Sharding is NOT just a one-time decision.
 *     It keeps affecting future operations.
 *
 *  3. CROSS-SHARD QUERIES BECOME PAINFUL ⚠️ (Biggest practical downside)
 *     If related data lives in different shards:
 *       → Queries that were once easy become complicated
 *       → Can't use simple local database joins
 *       → May need:
 *           Multiple queries
 *           Application-side joins
 *           Orchestration logic
 *           More latency
 *           More failure cases
 *
 *  4. OPERATIONAL COMPLEXITY INCREASES A LOT
 *     You now have to manage:
 *       → Shard health
 *       → Routing
 *       → Migrations
 *       → Skew detection
 *       → Rebalancing
 *       → Failures by shard
 *       → Schema coordination across shards
 *       → Maintenance windows across distributed topology
 *
 *     SHARDING IS OPS-HEAVY.
 *     It looks cool on a whiteboard — but comes with REAL BURDEN.
 */


/**
 * Part-8: SHARD KEY
 */

/**
 * Shard Key — Sabse Important Design Decision
 * ─────────────────────────────────────────────
 *
 *   Shard key decide karta hai ki konsa row konse shard mein jaayega.
 *   Ye sharded system ka MOST CRITICAL design choice hai.
 *   Galat shard key = sharding ka faayda khatam.
 *
 * WHAT IS A SHARD KEY?
 *   The field (or set of fields) whose value decides which shard a row belongs to.
 *
 *   Once you pick the shard key, you decide how:
 *   → Data gets spread across shards
 *   → Traffic gets distributed
 *
 * EXAMPLES OF SHARD KEYS:
 *   → country
 *   → user_id
 *   → (country, state)
 *   → first character of user identifier
 *   → hashed user_id
 *   → customer_id
 *
 * WHAT A GOOD SHARD KEY SHOULD DO:
 *
 *   1. DISTRIBUTE DATA AND TRAFFIC EVENLY
 *      You do NOT want one shard overloaded while others are idle.
 *
 *   2. MATCH COMMON QUERY PATTERNS
 *      If common queries naturally include the shard key,
 *      routing becomes easier and more efficient.
 *
 *   3. AVOID HOT SHARDS
 *      A hot shard gets far more traffic than others.
 *      That defeats the purpose of sharding.
 *
 *   4. REMAIN STABLE OVER TIME
 *      Changing shard logic later is EXPENSIVE.
 *      The initial choice matters A LOT.
 *
 * EXAMPLE OF A BAD SHARD KEY:
 *   Product has:
 *     → 85% of traffic from India
 *     → 15% of traffic from US
 *
 *   Choose country as shard key:
 *     → India ⇒ shard 1
 *     → US    ⇒ shard 2
 *
 *   THIS IS BAD.
 *   Shard 1 becomes a HOT SHARD.
 *   Most data and traffic pile onto one shard.
 *   The other shard remains underused.
 *
 *   The problem is not "we chose a key."
 *   The problem is "we chose a key that did NOT distribute traffic evenly."
 */


/**
 * Part-9: SHARD SKEW
 */

/**
 * Shard Skew — Jab Distribution Bigad Jaaye
 * ────────────────────────────────────────────
 *
 *   Shard skew = kuch shards pe bahut zyada load, baaki pe bahut kam.
 *   Overloaded shard naya bottleneck ban jaata hai — sharding ka faayda khatam.
 *
 * WHAT IS SHARD SKEW?
 *   Shard skew happens when some shards receive too much data or traffic
 *   compared to others.
 *
 * WHY SHARD SKEW IS DANGEROUS:
 *   The overloaded shard becomes the NEW BOTTLENECK.
 *   Even though the system is "sharded," one shard may behave
 *   like the old overloaded primary.
 *
 * VISUAL EXAMPLE:
 *
 *   GOOD Distribution (even):
 *   ┌─────────────────────────────────────────────────────────┐
 *   │ Shard 1: ████████ 33%                                   │
 *   │ Shard 2: ████████ 33%                                   │
 *   │ Shard 3: ████████ 34%                                   │
 *   └─────────────────────────────────────────────────────────┘
 *
 *   BAD Distribution (skewed):
 *   ┌─────────────────────────────────────────────────────────┐
 *   │ Shard 1: ████████████████████████████████ 85% ← HOT!    │
 *   │ Shard 2: ███ 15%                                        │
 *   │ Shard 3: █ 0%                                           │
 *   └─────────────────────────────────────────────────────────┘
 *
 * WHY CHANGING THE SHARD KEY LATER IS EXPENSIVE:
 *   If your shard key turns out wrong later, fixing it is PAINFUL:
 *   → Huge amounts of data already distributed
 *   → Routing logic depends on current key
 *   → Rebalancing means real migration work
 *   → System may need downtime or complicated live transfer
 *
 *   That is why the choice must be made with:
 *   → Data distribution in mind
 *   → Traffic distribution in mind
 *   → Query patterns in mind
 *   → Future scale in mind
 */


/**
 * Part-10: PLANNING FOR SHARDING
 */

/**
 * Ceiling Hit Hone Se Pehle Plan Karo
 * ──────────────────────────────────────
 *
 *   Wait mat karo system completely break hone tak.
 *   Sharding mein design, planning, data movement, testing sab lagta hai.
 *   Jab machine almost full ho jaaye — toh bahut der ho jaati hai.
 *
 * A VERY IMPORTANT PRACTICAL POINT:
 *   You do NOT wait until the system completely breaks
 *   to start thinking about sharding.
 *
 * WHY?
 *   Once the largest comfortable machine is nearly exhausted,
 *   you may already be TOO LATE to do the migration calmly.
 *
 * SHARDING TAKES:
 *   → Design
 *   → Planning
 *   → Data movement
 *   → Routing updates
 *   → Testing
 *   → Operational coordination
 *
 * THE HEALTHY PROGRESSION (Complete Journey):
 *
 *   STEP 1: OPTIMIZE
 *     → Schema design, Queries, Indexes
 *
 *   STEP 2: VERTICAL SCALE
 *     → Larger CPU, more memory, faster disk
 *
 *   STEP 3: READ REPLICAS
 *     → Scale reads, protect primary
 *
 *   STEP 4: PARTITIONING
 *     → Improve efficiency within one DB
 *
 *   STEP 5: SHARDING
 *     → When writes or storage outgrow single machine
 *     → This is a MAJOR architectural shift
 *
 *   Teams usually ANTICIPATE the need before the ceiling is actually hit.
 *   That is why good analytics, monitoring, and forward planning matter A LOT.
 */


In [ ]:
/**
 * Part-3: SHARDING VS REPLICATION
 */

In [ ]:
/**
 * Copy vs Divide — Simple Way to Remember
 * ──────────────────────────────────────────
 *
 *   REPLICAS COPY karte hain (same data everywhere)
 *   SHARDS DIVIDE karte hain (different data in each)
 *   Use case bhi alag hai — reads vs writes+storage.
 *
 * ┌─────────────────┬────────────────────────┬────────────────────────────┐
 * │ ASPECT          │ REPLICATION            │ SHARDING                   │
 * ├─────────────────┼────────────────────────┼────────────────────────────┤
 * │ What happens    │ Copy same data to      │ Split data across          │
 * │                 │ multiple DBs           │ multiple DBs               │
 * ├─────────────────┼────────────────────────┼────────────────────────────┤
 * │ Data in each DB │ All rows (full dataset)│ Only a slice               │
 * ├─────────────────┼────────────────────────┼────────────────────────────┤
 * │ Primary purpose │ Scale reads            │ Scale writes and storage   │
 * ├─────────────────┼────────────────────────┼────────────────────────────┤
 * │ Write capacity  │ Still limited by       │ Distributed across         │
 * │                 │ single primary         │ multiple shards            │
 * ├─────────────────┼────────────────────────┼────────────────────────────┤
 * │ Storage         │ One primary holds all  │ Distributed across shards  │
 * ├─────────────────┼────────────────────────┼────────────────────────────┤
 * │ Complexity      │ Lower                  │ Much higher                │
 * └─────────────────┴────────────────────────┴────────────────────────────┘
 *
 * WHEN TO USE EACH:
 *   Replication  → when READS are the bottleneck
 *   Sharding     → when WRITES or STORAGE become the bottleneck
 */

In [ ]:
/**
 * Part-4: SHARDING VS PARTITIONING
 */

In [ ]:
/**
 * Ek DB Ke Andar vs Multiple DBs
 * ────────────────────────────────
 *
 *   Partitioning = ek hi database ke andar data ko slices mein divide karna.
 *   Sharding = multiple databases ke across data divide karna.
 *   Dono alag problems solve karte hain.
 *
 * PARTITIONING:
 *   → Happens INSIDE a single database instance
 *   → Managed by the database engine itself
 *   → Used to improve query efficiency
 *   → Still ONE database server
 *   → Easier joins and coordination
 *   → Simpler operationally
 *
 * SHARDING:
 *   → Happens ACROSS multiple database instances
 *   → Requires application/routing logic
 *   → Used to scale writes and storage
 *   → MULTIPLE database servers
 *   → Cross-shard operations become hard
 *   → Much higher operational complexity
 *
 * USE PARTITIONING WHEN:
 *   → Data still fits within one server
 *   → Queries are slow because they scan too much data
 *   → You want better efficiency without moving to multiple DBs
 *
 * USE SHARDING WHEN:
 *   → One server is no longer enough
 *   → Storage or write load has outgrown single-machine limits
 *   → True horizontal scale is needed
 *
 *   Partitioning improves organization INSIDE one DB.
 *   Sharding changes the WHOLE architecture.
 */

In [ ]:
/**
 * Part-5: PRACTICAL EXAMPLE
 */

In [ ]:
/**
 * Orders Table Ko Country Ke Hisaab Se Shard Karna
 * ────────────────────────────────────────────────────
 *
 *   Company USA mein start hoti hai, phir India aur Europe expand karti hai.
 *   Orders table ENORMOUS ho jaati hai — billions of rows.
 *   Even replicas ke baad bhi ek writer kaafi nahi.
 *   Solution: country ke hisaab se data shard karo.
 *
 * THE SCENARIO:
 *   → Company starts in USA
 *   → Expands to India, Europe, other regions
 *   → Orders table: billions of rows
 *   → Writes happening 24/7 from many countries
 *   → Even after optimization and replicas — ONE WRITER is not enough
 *
 * BEFORE SHARDING (one database has everything):
 *
 *   ┌──────────────────────────────────────────────────────────┐
 *   │                    ORDERS TABLE                          │
 *   │   US order 1, US order 2                                 │
 *   │   India order 1, India order 2                           │
 *   │   EU order 1, EU order 2                                 │
 *   │   ... billions more ...                                  │
 *   └──────────────────────────────────────────────────────────┘
 *
 * AFTER SHARDING (split by country):
 *
 *   ┌──────────────────┐  ┌──────────────────┐  ┌──────────────────┐
 *   │   SHARD 1        │  │   SHARD 2        │  │   SHARD 3        │
 *   │   US orders only │  │   India orders   │  │   EU orders only │
 *   │                  │  │   only           │  │                  │
 *   │   US order 1     │  │   India order 1  │  │   EU order 1     │
 *   │   US order 2     │  │   India order 2  │  │   EU order 2     │
 *   │   ...            │  │   ...            │  │   ...            │
 *   └──────────────────┘  └──────────────────┘  └──────────────────┘
 *
 * WHAT THIS ACHIEVES:
 *   → US write traffic    → hits shard 1
 *   → India write traffic → hits shard 2
 *   → EU write traffic    → hits shard 3
 *
 *   Result:
 *   → Write load no longer concentrated in one place
 *   → Storage is distributed
 *   → More storage runway
 *   → More write runway
 */

In [ ]:
/**
 * Part-6: PROS OF SHARDING
 */

In [ ]:
/**
 * Sharding ke Benefits
 * ──────────────────────
 *
 *   Sharding kab beneficial hota hai? Ye 5 main advantages hain.
 *
 *  1. MORE STORAGE RUNWAY
 *     Ek database ko sab kuch hold nahi karna.
 *     Data spread across shards → total storage capacity significantly increases.
 *
 *  2. MORE WRITE RUNWAY
 *     Write traffic distributed across shards.
 *     One node no longer handles every write.
 *     This is one of the BIGGEST reasons sharding exists.
 *
 *  3. ONE NODE NO LONGER CARRIES EVERYTHING
 *     Operationally ye bahut help karta hai.
 *     One overloaded node no longer responsible for full system scale.
 *     Pressure ka concentration reduce hota hai.
 *
 *  4. BETTER FAULT ISOLATION
 *     If one shard has a problem:
 *       → Only part of the data is affected
 *       → Only one segment of traffic is impacted
 *       → NOT necessarily the whole world
 *
 *     MUCH better than one giant all-powerful primary
 *     being the only place everything lives.
 *
 *  5. GEOGRAPHICAL OPTIMIZATION
 *     Shards can be placed closer to where data originates.
 *     Example: India shard in India → lower latency for Indian users.
 */

In [ ]:
/**
 * Part-7: CONS OF SHARDING
 */

In [ ]:
/**
 * Sharding ke Complexity aur Tradeoffs
 * ──────────────────────────────────────
 *
 *   Sharding whiteboard pe cool lagta hai — lekin real mein bahut bhaaree hai.
 *   Ye 4 major downsides hain jo production mein feel hote hain.
 *
 *  1. ROUTING LOGIC BECOMES YOUR RESPONSIBILITY
 *     Data split hone ke baad application/routing layer ko PATA HONA CHAHIYE:
 *       → Which shard to query
 *       → Which shard to write to
 *       → How to find the right shard for a given request
 *
 *     That logic must be designed, implemented, tested, and maintained.
 *
 *  2. REBALANCING IS PAINFUL
 *     Suppose one shard becomes too large or too hot.
 *     Now you may need to move data AGAIN:
 *       → Repartitioning
 *       → Migration
 *       → Downtime or careful live migration
 *       → More routing changes
 *       → Lots of operational work
 *
 *     Sharding is NOT just a one-time decision.
 *     It keeps affecting future operations.
 *
 *  3. CROSS-SHARD QUERIES BECOME PAINFUL ⚠️ (Biggest practical downside)
 *     If related data lives in different shards:
 *       → Queries that were once easy become complicated
 *       → Can't use simple local database joins
 *       → May need:
 *           Multiple queries
 *           Application-side joins
 *           Orchestration logic
 *           More latency
 *           More failure cases
 *
 *  4. OPERATIONAL COMPLEXITY INCREASES A LOT
 *     You now have to manage:
 *       → Shard health
 *       → Routing
 *       → Migrations
 *       → Skew detection
 *       → Rebalancing
 *       → Failures by shard
 *       → Schema coordination across shards
 *       → Maintenance windows across distributed topology
 *
 *     SHARDING IS OPS-HEAVY.
 *     It looks cool on a whiteboard — but comes with REAL BURDEN.
 */

In [ ]:
/**
 * Part-8: SHARD KEY
 */

In [ ]:
/**
 * Shard Key — Sabse Important Design Decision
 * ─────────────────────────────────────────────
 *
 *   Shard key decide karta hai ki konsa row konse shard mein jaayega.
 *   Ye sharded system ka MOST CRITICAL design choice hai.
 *   Galat shard key = sharding ka faayda khatam.
 *
 * WHAT IS A SHARD KEY?
 *   The field (or set of fields) whose value decides which shard a row belongs to.
 *
 *   Once you pick the shard key, you decide how:
 *   → Data gets spread across shards
 *   → Traffic gets distributed
 *
 * EXAMPLES OF SHARD KEYS:
 *   → country
 *   → user_id
 *   → (country, state)
 *   → first character of user identifier
 *   → hashed user_id
 *   → customer_id
 *
 * WHAT A GOOD SHARD KEY SHOULD DO:
 *
 *   1. DISTRIBUTE DATA AND TRAFFIC EVENLY
 *      You do NOT want one shard overloaded while others are idle.
 *
 *   2. MATCH COMMON QUERY PATTERNS
 *      If common queries naturally include the shard key,
 *      routing becomes easier and more efficient.
 *
 *   3. AVOID HOT SHARDS
 *      A hot shard gets far more traffic than others.
 *      That defeats the purpose of sharding.
 *
 *   4. REMAIN STABLE OVER TIME
 *      Changing shard logic later is EXPENSIVE.
 *      The initial choice matters A LOT.
 *
 * EXAMPLE OF A BAD SHARD KEY:
 *   Product has:
 *     → 85% of traffic from India
 *     → 15% of traffic from US
 *
 *   Choose country as shard key:
 *     → India ⇒ shard 1
 *     → US    ⇒ shard 2
 *
 *   THIS IS BAD.
 *   Shard 1 becomes a HOT SHARD.
 *   Most data and traffic pile onto one shard.
 *   The other shard remains underused.
 *
 *   The problem is not "we chose a key."
 *   The problem is "we chose a key that did NOT distribute traffic evenly."
 */

In [ ]:
/**
 * Part-9: SHARD SKEW
 */

In [ ]:
/**
 * Shard Skew — Jab Distribution Bigad Jaaye
 * ────────────────────────────────────────────
 *
 *   Shard skew = kuch shards pe bahut zyada load, baaki pe bahut kam.
 *   Overloaded shard naya bottleneck ban jaata hai — sharding ka faayda khatam.
 *
 * WHAT IS SHARD SKEW?
 *   Shard skew happens when some shards receive too much data or traffic
 *   compared to others.
 *
 * WHY SHARD SKEW IS DANGEROUS:
 *   The overloaded shard becomes the NEW BOTTLENECK.
 *   Even though the system is "sharded," one shard may behave
 *   like the old overloaded primary.
 *
 * VISUAL EXAMPLE:
 *
 *   GOOD Distribution (even):
 *   ┌─────────────────────────────────────────────────────────┐
 *   │ Shard 1: ████████ 33%                                   │
 *   │ Shard 2: ████████ 33%                                   │
 *   │ Shard 3: ████████ 34%                                   │
 *   └─────────────────────────────────────────────────────────┘
 *
 *   BAD Distribution (skewed):
 *   ┌─────────────────────────────────────────────────────────┐
 *   │ Shard 1: ████████████████████████████████ 85% ← HOT!    │
 *   │ Shard 2: ███ 15%                                        │
 *   │ Shard 3: █ 0%                                           │
 *   └─────────────────────────────────────────────────────────┘
 *
 * WHY CHANGING THE SHARD KEY LATER IS EXPENSIVE:
 *   If your shard key turns out wrong later, fixing it is PAINFUL:
 *   → Huge amounts of data already distributed
 *   → Routing logic depends on current key
 *   → Rebalancing means real migration work
 *   → System may need downtime or complicated live transfer
 *
 *   That is why the choice must be made with:
 *   → Data distribution in mind
 *   → Traffic distribution in mind
 *   → Query patterns in mind
 *   → Future scale in mind
 */

In [ ]:
/**
 * Part-10: PLANNING FOR SHARDING
 */

In [ ]:
/**
 * Ceiling Hit Hone Se Pehle Plan Karo
 * ──────────────────────────────────────
 *
 *   Wait mat karo system completely break hone tak.
 *   Sharding mein design, planning, data movement, testing sab lagta hai.
 *   Jab machine almost full ho jaaye — toh bahut der ho jaati hai.
 *
 * A VERY IMPORTANT PRACTICAL POINT:
 *   You do NOT wait until the system completely breaks
 *   to start thinking about sharding.
 *
 * WHY?
 *   Once the largest comfortable machine is nearly exhausted,
 *   you may already be TOO LATE to do the migration calmly.
 *
 * SHARDING TAKES:
 *   → Design
 *   → Planning
 *   → Data movement
 *   → Routing updates
 *   → Testing
 *   → Operational coordination
 *
 * THE HEALTHY PROGRESSION (Complete Journey):
 *
 *   STEP 1: OPTIMIZE
 *     → Schema design, Queries, Indexes
 *
 *   STEP 2: VERTICAL SCALE
 *     → Larger CPU, more memory, faster disk
 *
 *   STEP 3: READ REPLICAS
 *     → Scale reads, protect primary
 *
 *   STEP 4: PARTITIONING
 *     → Improve efficiency within one DB
 *
 *   STEP 5: SHARDING
 *     → When writes or storage outgrow single machine
 *     → This is a MAJOR architectural shift
 *
 *   Teams usually ANTICIPATE the need before the ceiling is actually hit.
 *   That is why good analytics, monitoring, and forward planning matter A LOT.
 */